[//]: # (cr:doc name='chapter_11_scoring_validation_explanations' id=b27a29b8)
# Chapter 11: Scoring, Validation & Explanations

End-to-end scoring pipeline with holdout validation, model comparison, adversarial
validation, SHAP explanations, and error analysis.

**Sections:**
1. Run Scoring
2. Summary Metrics
3. Model Comparison Grid
4. Adversarial Pipeline Validation
5. Transformation Validation
6. Model Explanations (SHAP)
7. Customer Browser
8. Error Analysis
9. Export Results

In [ ]:
# @cr:code name='init_progress' id=ec249a4c
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous
from customer_retention.analysis.visualization import display_table

accept_workflow_params()
track_and_export_previous("11_scoring_validation.ipynb")

import sys
from pathlib import Path

from customer_retention.core.compat import native_pd, to_pandas
from customer_retention.core.config.experiments import (  # noqa: F401
    FINDINGS_DIR,
    OUTPUT_DIR,
    setup_experiments_structure,
)

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='load_scoring_config' id=a6a4acfd
from customer_retention.core.compat.detection import is_databricks
from customer_retention.stages.scoring import ScoringConfig, ScoringDataLoader

if is_databricks():
    try:
        config = ScoringConfig.from_databricks()
    except ValueError as e:
        import os
        print(f"ERROR: {e}")
        print("\nDiagnostic info:")
        print(f"  CR_EXPERIMENT_NAME = {os.environ.get('CR_EXPERIMENT_NAME', '(not set)')}")
        print(f"  CR_CATALOG = {os.environ.get('CR_CATALOG', '(not set)')}")
        print(f"  CR_SCHEMA = {os.environ.get('CR_SCHEMA', '(not set)')}")
        print("\nOn Databricks, experiments are created under /Users/{username}/.")
        print("Set CR_EXPERIMENT_NAME to the full path, e.g.:")
        print('  os.environ["CR_EXPERIMENT_NAME"] = "/Users/you@example.com/customer_churn"')
        raise
else:
    generated_dir = Path("../generated_pipelines/local")
    pipeline_dirs = sorted(generated_dir.glob("*/config.py"))
    if not pipeline_dirs:
        raise FileNotFoundError(
            f"No generated pipeline found under {generated_dir}. Run notebook 10 first."
        )
    config = ScoringConfig.from_local_config(pipeline_dirs[-1].parent)

loader = ScoringDataLoader(config)

PIPELINE_NAME = config.pipeline_name
TARGET_COLUMN = config.target_column
ENTITY_KEY = config.entity_key
RECOMMENDATIONS_HASH = config.recommendations_hash
ORIGINAL_COLUMN = config.original_column

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Platform: {'Databricks' if config.is_databricks else 'Local'}")
print(f"Experiments dir: {config.experiments_dir}")
print(f"Recommendations hash: {RECOMMENDATIONS_HASH}")

In [ ]:
# @cr:code name='display_pipeline_metadata' id=9d2d2a44
import json as _json

from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace

_ns = RunNamespace.from_env_or_latest(config.experiments_dir)
_stage_meta = {}
_exploration_diag = None
if _ns is not None:
    for _stage, _path in [
        ('bronze', _ns.bronze_metadata_path),
        ('silver', _ns.silver_metadata_path),
        ('gold', _ns.gold_metadata_path),
        ('training', _ns.training_metadata_path),
    ]:
        if _path.exists():
            _stage_meta[_stage] = _json.loads(_path.read_text())
    if _ns.exploration_diagnostics_path.exists():
        _exploration_diag = _json.loads(_ns.exploration_diagnostics_path.read_text())

if _stage_meta:
    print('=' * 50)
    print('PIPELINE STAGE SUMMARY')
    print('=' * 50)
    if 'bronze' in _stage_meta:
        bm = _stage_meta['bronze']
        print(f'\nBronze: {bm.get("total_sources", 0)} sources')
        for name, info in bm.get('sources', {}).items():
            if isinstance(info, dict):
                print(f'  {name}: {info.get("rows", "?"):,} rows, {info.get("columns", "?")} columns')
            else:
                print(f'  {name}: {info}')
    if 'silver' in _stage_meta:
        sm = _stage_meta['silver']
        print(f'\nSilver: {sm.get("rows", "?"):,} rows, {sm.get("columns", "?")} columns')
        if 'elapsed_seconds' in sm:
            print(f'  Elapsed: {sm["elapsed_seconds"]}s')
        if 'source_count' in sm:
            print(f'  Sources merged: {sm["source_count"]}')
    if 'gold' in _stage_meta:
        gm = _stage_meta['gold']
        print(f'\nGold: {gm.get("rows", "?"):,} rows, {gm.get("columns", "?")} columns')
        if 'feature_count' in gm:
            print(f'  Features: {gm["feature_count"]}')
        if 'elapsed_seconds' in gm:
            print(f'  Elapsed: {gm["elapsed_seconds"]}s')
    if 'training' in _stage_meta:
        tm = _stage_meta['training']
        print('\nTraining:')
        if 'best_model' in tm:
            print(f'  Best model: {tm["best_model"]}')
        if 'best_roc_auc' in tm:
            print(f'  Best ROC-AUC: {float(tm["best_roc_auc"]):.4f}')
        if 'split' in tm:
            sp = tm['split']
            print(f'  Train/test: {sp.get("train", "?"):,} / {sp.get("test", "?"):,}')
            if 'cutoff_date' in sp:
                print(f'  Temporal cutoff: {sp["cutoff_date"]}')
else:
    print('No pipeline metadata found. Run the pipeline (notebook 10) first.')

if _exploration_diag:
    print(f'\nExploration diagnostics loaded (best model: {_exploration_diag.get("best_model_name", "?")})')


[//]: # (cr:doc name='11_1_run_scoring' id=0f02f66b)
## 11.1 Run Scoring

In [ ]:
# @cr:code name='import_mlflow' id=ee54d054
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import xgboost as xgb

from customer_retention.core.compat import track_stage_object
from customer_retention.transforms import TransformExecutor

_executor = TransformExecutor()
_registry = loader.load_artifact_store()
ENCODINGS, SCALINGS = loader.load_transforms()

PREDICTIONS_PATH = config.production_dir / "data" / "scoring" / "predictions.parquet"

mlflow.set_tracking_uri(config.mlflow_tracking_uri)

features_df = loader.load_gold_features_distributed()
track_stage_object(features_df)

_SCORING_AVAILABLE = True

if ORIGINAL_COLUMN not in features_df.columns:
    print(
        f"No holdout found (column '{ORIGINAL_COLUMN}' missing). "
        "Holdout must be created in silver layer BEFORE gold layer feature computation.\n"
        "Skipping scoring validation."
    )
    _SCORING_AVAILABLE = False
    predictions_df = native_pd.DataFrame(columns=[ENTITY_KEY, "prediction", "probability", "actual", "correct"])

if _SCORING_AVAILABLE:
    scoring_mask = features_df[TARGET_COLUMN].isna() & features_df[ORIGINAL_COLUMN].notna()
    scoring_df = to_pandas(features_df[scoring_mask]).copy()
    print(f"Found {len(scoring_df):,} holdout records for scoring")

    scoring_features = loader.load_scoring_features(scoring_df)

    model, model_uri = loader.load_model()
    print(f"Loading model: {model_uri}")

    _is_spark_ml = hasattr(model, "transform") and not hasattr(model, "predict_proba")

    _SCORING_AVAILABLE = True


    def prepare_features(df):
        return loader.prepare_features(df, ENCODINGS + SCALINGS, _executor, _registry)


    X = prepare_features(scoring_features)
    y_true = scoring_features[ORIGINAL_COLUMN].to_numpy()

    if X.shape[1] == 0:
        print(
            "WARNING: Feature matrix has 0 columns after preparation.\n"
            "The model was likely trained before feature selection was fixed.\n"
            "Re-run notebooks 08 and 10 to retrain, then re-run this notebook.\n"
            "Skipping scoring validation."
        )
        _SCORING_AVAILABLE = False
        predictions_df = native_pd.DataFrame(columns=[ENTITY_KEY, "prediction", "probability", "actual", "correct"])

    if _SCORING_AVAILABLE:
        _training_features = loader.load_training_feature_names()
        _scoring_cols = set(X.columns)
        _train_set = set(_training_features)
        _missing_feats = sorted(_train_set - _scoring_cols)
        _extra_feats = sorted(_scoring_cols - _train_set)
        if _missing_feats or _extra_feats:
            print("=" * 70)
            print("FEATURE MISMATCH: model training vs scoring data")
            print("=" * 70)
            if _missing_feats:
                print(f"\n  MISSING in scoring ({len(_missing_feats)} features the model expects but scoring data lacks):")
                for f in _missing_feats:
                    print(f"    - {f}")
            if _extra_feats:
                print(f"\n  EXTRA in scoring ({len(_extra_feats)} features in scoring data but not in model):")
                for f in _extra_feats:
                    print(f"    - {f}")
            print(f"\n  Model trained with: {len(_training_features)} features")
            print(f"  Scoring data has:   {len(_scoring_cols)} features")
            print("\n  This usually means the model was trained in a previous run with")
            print("  different feature selection. Re-run NB08 + NB10 to retrain.")
            print("=" * 70)
            raise ValueError(
                f"Feature mismatch: model expects {len(_training_features)} features, "
                f"scoring data has {len(_scoring_cols)} ({len(_missing_feats)} missing, {len(_extra_feats)} extra). "
                "Re-run NB08 + NB10 to retrain."
            )
        print(f"Feature check: model and scoring data aligned ({len(_training_features)} features)")

    if _SCORING_AVAILABLE:
        print("Generating predictions...")
        if _is_spark_ml:
            y_proba = loader.predict_spark_ml(model, X, feature_names=_training_features)
        elif hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X[_training_features])[:, 1]
        else:
            y_proba = model.predict(xgb.DMatrix(X[_training_features], feature_names=_training_features))
        y_pred = (y_proba >= 0.5).astype(int)

        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0,
        }
        print("\nValidation Metrics (vs original values):")
        for name, value in metrics.items():
            print(f"  {name}: {value:.4f}")

        predictions_df = native_pd.DataFrame({
            ENTITY_KEY: scoring_df[ENTITY_KEY].to_numpy(),
            "prediction": y_pred,
            "probability": y_proba,
            "actual": y_true,
            "correct": (y_pred == y_true).astype(int),
        })
        PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
        predictions_df.to_parquet(PREDICTIONS_PATH, index=False)
        print(f"\nPredictions saved: {PREDICTIONS_PATH}")
        print(f"Correct: {predictions_df['correct'].sum():,}/{len(predictions_df):,} ({predictions_df['correct'].mean():.1%})")

[//]: # (cr:doc name='11_2_summary_metrics' id=457c6ec3)
## 11.2 Summary Metrics

In [ ]:
# @cr:code name='import_metrics' id=01354e15
if _SCORING_AVAILABLE:
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    y_true = predictions_df["actual"]
    y_pred = predictions_df["prediction"]
    y_proba = predictions_df["probability"]

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0,
    }

    print("\n=== Scoring Validation Metrics ===")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")
else:
    print("Sections 11.2-11.9 skipped: retrain model via notebooks 08 + 10")

In [ ]:
# @cr:code name='plot_roc_curve' id=2a6bef25
if _SCORING_AVAILABLE:
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, "b-", lw=2, label=f"ROC (AUC={metrics['ROC-AUC']:.3f})")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve")
    axes[0].legend()

    axes[1].hist(y_proba[y_true == 0], bins=30, alpha=0.5, label="Actual=0", color="blue")
    axes[1].hist(y_proba[y_true == 1], bins=30, alpha=0.5, label="Actual=1", color="red")
    axes[1].axvline(x=0.5, color="black", linestyle="--", label="Threshold")
    axes[1].set_xlabel("Predicted Probability")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Probability Distribution")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

[//]: # (cr:doc name='11_3_model_comparison_grid' id=faaa07a9)
## 11.3 Production Model Card

Compare the production-trained model against the exploration baseline.
Shows training context (dataset size, temporal boundaries) and side-by-side metrics.


In [ ]:
# @cr:code name='production_model_card' id=f824276e
if _SCORING_AVAILABLE:
    from IPython.display import Markdown, display
    from sklearn.metrics import average_precision_score

    _tm = _stage_meta.get('training', {})
    _exploration_holdout = (_exploration_diag or {}).get('best_model_holdout_metrics')

    # --- Model Card ---
    display(Markdown('### Production Model Card'))
    _gold_data = _tm.get('gold_data', {})
    _split = _tm.get('split', {})
    _card_rows = [
        ('Model Type', _tm.get('best_model', 'unknown')),
        ('Dataset Size', f'{_gold_data.get("rows", "?"):,} rows x {_tm.get("feature_count", "?")} features'),
        ('Train / Test Split', f'{_split.get("train", "?"):,} / {_split.get("test", "?"):,}'),
        ('Temporal Cutoff', _split.get('cutoff_date', 'N/A')),
        ('Production ROC-AUC (test)', f'{float(_tm.get("best_roc_auc", 0)):.4f}'),
    ]
    if _exploration_holdout:
        _card_rows.append(('Exploration Best Model', _exploration_holdout.get('model_name', '?')))
        _card_rows.append(('Exploration ROC-AUC (test)', f'{_exploration_holdout.get("roc_auc", 0):.4f}'))
    display_table(native_pd.DataFrame(_card_rows, columns=['Property', 'Value']))

    # --- Exploration vs Production Metrics ---
    _prod_best = _tm.get('best_model', 'unknown')
    _prod_metrics = _tm.get('models', {}).get(_prod_best, {})

    _holdout_roc_auc = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0
    _holdout_pr_auc = average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0

    _metrics_keys = ['roc_auc', 'pr_auc', 'f1', 'precision', 'recall', 'accuracy']
    _comparison_rows = []
    for _mk in _metrics_keys:
        _exp_val = _exploration_holdout.get(_mk) if _exploration_holdout else None
        _prod_val = _prod_metrics.get(_mk)
        _comparison_rows.append({
            'Metric': _mk.replace('_', ' ').title(),
            'Exploration (test)': f'{_exp_val:.4f}' if _exp_val is not None else 'N/A',
            'Production (test)': f'{_prod_val:.4f}' if _prod_val is not None else 'N/A',
            'Holdout': f'{metrics.get(_mk, 0):.4f}' if _mk in metrics else 'N/A',
            'Delta (prod-exp)': f'{(_prod_val or 0) - (_exp_val or 0):+.4f}' if _exp_val and _prod_val else '',
        })

    display(Markdown('### Exploration vs Production vs Holdout'))
    display_table(native_pd.DataFrame(_comparison_rows))

    # Highlight holdout vs production test
    _prod_test_auc = _prod_metrics.get('roc_auc', 0)
    _holdout_delta = _holdout_roc_auc - _prod_test_auc
    if abs(_holdout_delta) > 0.05:
        print(f'\nWARNING: Holdout ROC-AUC ({_holdout_roc_auc:.4f}) differs from production test ({_prod_test_auc:.4f}) by {_holdout_delta:+.4f}')
    else:
        print(f'\nHoldout ROC-AUC ({_holdout_roc_auc:.4f}) is consistent with production test ({_prod_test_auc:.4f})')


In [ ]:
# @cr:code name='plot_holdout_performance' id=d95a7a8a
if _SCORING_AVAILABLE:
    from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    ax = axes[0]
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['Actual 0', 'Actual 1'])
    for i in range(2):
        for j in range(2):
            pct = cm[i, j] / cm.sum() * 100
            ax.text(j, i, f'{cm[i, j]}\n({pct:.1f}%)', ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=10)
    ax.set_title(f'{_tm.get("best_model", "Production")}\nAccuracy: {metrics.get("accuracy", metrics.get("Accuracy", 0)):.3f}', fontweight='bold')

    # ROC Curve
    ax = axes[1]
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc_val = roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0
    ax.plot(fpr, tpr, 'b-', lw=2, label=f'Production (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.fill_between(fpr, tpr, alpha=0.15, color='blue')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

    # PR Curve
    ax = axes[2]
    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_proba)
    pr_auc_val = average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0
    ax.plot(recall_vals, precision_vals, 'b-', lw=2, label=f'Production (PR-AUC={pr_auc_val:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
# @cr:code name='feature_comparison' id=0e84bb4b
if _SCORING_AVAILABLE:
    _exp_features = set()
    _prod_features = set(_training_features)

    if _exploration_diag:
        _exp_features = set(_exploration_diag.get('feature_names', []))

    _common = _exp_features & _prod_features
    _exp_only = sorted(_exp_features - _prod_features)
    _prod_only = sorted(_prod_features - _exp_features)

    display(Markdown('### Feature Set Comparison'))
    _feat_summary = [
        ('Exploration features', len(_exp_features)),
        ('Production features', len(_prod_features)),
        ('Common', len(_common)),
        ('Exploration-only', len(_exp_only)),
        ('Production-only', len(_prod_only)),
    ]
    display_table(native_pd.DataFrame(_feat_summary, columns=['', 'Count']))

    if _exp_only:
        _excluded = _stage_meta.get('training', {}).get('feature_profile', {}).get('excluded_details', {})
        _reasons = []
        for f in _exp_only:
            reason = _excluded.get(f, 'unknown')
            _reasons.append({'Feature': f, 'Exclusion Reason': reason})
        display(Markdown('### Features in Exploration but Not Production'))
        display_table(native_pd.DataFrame(_reasons[:30]))

    if _prod_only:
        print(f'\nFeatures in production but not exploration: {_prod_only[:20]}')


[//]: # (cr:doc name='11_4_adversarial_pipeline_validation' id=49274fc4)
## 11.4 Adversarial Pipeline Validation

Validate that scoring pipeline produces identical features to training for holdout entities.
This catches transformation inconsistencies (e.g., scalers re-fit, encoders handling unseen values differently).

In [ ]:
# @cr:code name='evaluate_holdout' id=1c4bb5db
if _SCORING_AVAILABLE:
    from customer_retention.stages.validation import AdversarialScoringValidator

    print('\n' + '=' * 60)
    print('ADVERSARIAL PIPELINE VALIDATION')
    print('=' * 60)

    adversarial_result = AdversarialScoringValidator(
        features_df, entity_column=ENTITY_KEY,
        target_column=TARGET_COLUMN, tolerance=1e-6,
    ).validate_features(scoring_features)

    print(adversarial_result.summary)

    if not adversarial_result.passed:
        # Load recommendation registry for feature-to-stage mapping
        _rec_registry = None
        if _ns and _ns.merged_recommendations_path.exists():
            from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
            _rec_registry = RecommendationRegistry.load(_ns.merged_recommendations_path)
        else:
            print('\nWARNING: No merged recommendations found — cannot group drifts by pipeline stage')

        if _rec_registry:
            _feature_stage_map = {}
            for _rec in _rec_registry.all_recommendations:
                _feature_stage_map[_rec.target_column] = {
                    'layer': _rec.layer,
                    'category': _rec.category,
                    'action': _rec.action,
                    'source_notebook': _rec.source_notebook,
                }

            from collections import defaultdict
            _grouped = defaultdict(list)
            for _drift in adversarial_result.feature_drifts:
                _stage_info = _feature_stage_map.get(_drift.feature_name, {'layer': 'unknown', 'category': 'unknown'})
                _key = f"{_stage_info['layer'].title()} / {_stage_info['category']}"
                _grouped[_key].append((_drift, _stage_info))

            display(Markdown('### Drift by Pipeline Stage'))
            for _group_name, _items in sorted(_grouped.items()):
                print(f'\n{_group_name} ({len(_items)} features)')
                for _drift, _info in _items:
                    _sev = _drift.severity.name
                    print(f'  [{_sev}] {_drift.feature_name}: max_diff={_drift.max_absolute_diff:.6f}, affected={_drift.affected_entities}/{adversarial_result.entities_validated}')
                    if _drift.sample_entity_values:
                        for _sv in _drift.sample_entity_values[:3]:
                            print(f'    entity {_sv["entity_id"]}: gold={_sv["gold_value"]:.4f} -> scoring={_sv["scoring_value"]:.4f}')
                    if _drift.gold_mean is not None:
                        print(f'    Distribution: gold mu={_drift.gold_mean:.4f} sigma={_drift.gold_std:.4f} | scoring mu={_drift.scoring_mean:.4f} sigma={_drift.scoring_std:.4f}')
                    _src = _info.get('source_notebook', '?')
                    _act = _info.get('action', '?')
                    if _src != '?':
                        print(f'    Source: {_src} -> {_act}')
        else:
            display_table(adversarial_result.to_dataframe())


[//]: # (cr:doc name='11_5_transformation_validation' id=e84b7ff6)
## 11.5 Transformation Validation

Use `validate_feature_transformation()` from the validation module to verify
encoding/scaling consistency between training and scoring.

In [ ]:
# @cr:code name='validate_feature_transform' id=25635009
if _SCORING_AVAILABLE:
    from customer_retention.stages.validation import validate_feature_transformation

    report = validate_feature_transformation(
        transform_fn=prepare_features, entity_column=ENTITY_KEY, verbose=True,
        gold_features=features_df, holdout_column=ORIGINAL_COLUMN,
        target_column=TARGET_COLUMN,
    )

    if report.passed:
        print("Transformation validation PASSED")
    else:
        print(f"Transformation validation FAILED: {len(report.feature_mismatches)} mismatches")


[//]: # (cr:doc name='11_6_model_explanations_shap' id=0158165b)
## 11.6 Model Explanations (SHAP)

In [ ]:
# @cr:code name='setup_shap' id=68e840d7
if _SCORING_AVAILABLE:
    import shap

    model, model_uri = loader.load_model()
    _is_spark_ml = config.is_databricks
    print(f"Loading model: {model_uri}")
    print(f"Model type: {type(model).__name__}")


In [ ]:
# @cr:config name='shap_sample_config' id=df072693
SHAP_SAMPLE_SIZE = 200
SHAP_BACKGROUND_SIZE = 100
SHAP_RISK_BINS = 5
SHAP_TOP_FEATURES = 15
SHAP_PRIORITY_CONDITION = None

In [ ]:
# @cr:code name='prepare_shap_features' id=4add2d0b
if _SCORING_AVAILABLE:
    from customer_retention.analysis.interpretability import select_risk_stratified_sample

    X = prepare_features(scoring_features)
    X = X[_training_features]
    feature_names = list(X.columns)
    print(f"Prepared {len(feature_names)} features for SHAP analysis")

    _priority_mask = None
    if SHAP_PRIORITY_CONDITION is not None:
        _priority_mask = SHAP_PRIORITY_CONDITION(scoring_features)

    _shap_indices = select_risk_stratified_sample(
        y_proba, n_samples=SHAP_SAMPLE_SIZE,
        n_bins=SHAP_RISK_BINS, priority_mask=_priority_mask,
    )
    X_shap = X.iloc[_shap_indices].reset_index(drop=True)
    print(f"Selected {len(X_shap)} / {len(X)} records for SHAP (risk-stratified)")
    _prob_bins = native_pd.cut(y_proba[_shap_indices], bins=SHAP_RISK_BINS)
    print(f"Sample distribution by risk bin:\n{_prob_bins.value_counts().sort_index()}")

In [ ]:
# @cr:code name='create_shap_explainer' id=54322b14
if _SCORING_AVAILABLE:
    print("Creating SHAP explainer...")

    _is_tree_model = (
        hasattr(model, "estimators_")
        or hasattr(model, "get_booster")
        or type(model).__name__ == "Booster"
    )

    if _is_tree_model:
        explainer = shap.TreeExplainer(model)
        print(f"Using TreeExplainer ({type(model).__name__})")
    else:
        _bg_size = min(SHAP_BACKGROUND_SIZE, len(X_shap))
        background = shap.sample(X_shap, _bg_size)
        _max_evals = 2 * len(feature_names) + 1
        if _is_spark_ml:
            def _predict_fn(x):
                return loader.predict_spark_ml(
                    model, native_pd.DataFrame(x, columns=feature_names), feature_names,
                )
        elif hasattr(model, "predict_proba"):
            _predict_fn = model.predict_proba
        else:
            _predict_fn = model
        explainer = shap.Explainer(
            _predict_fn, background,
            feature_names=feature_names, max_evals=_max_evals,
        )
        print(f"Using PermutationExplainer (max_evals={_max_evals})")

    print(f"Computing SHAP values for {len(X_shap)} sampled records...")
    shap_values = explainer(X_shap)
    print(f"SHAP values computed for {len(shap_values)} records")

In [ ]:
# @cr:code name='extract_shap_values' id=9e11c183
if _SCORING_AVAILABLE:
    if len(shap_values.shape) == 3:
        shap_vals = shap_values[:, :, 1]
    else:
        shap_vals = shap_values

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_vals, X_shap, feature_names=feature_names, show=False, max_display=SHAP_TOP_FEATURES)
    plt.title(f"Feature Importance (SHAP Summary, n={len(X_shap)})")
    plt.tight_layout()
    plt.show()

In [ ]:
# @cr:code name='rank_shap_importance' id=181959a1
if _SCORING_AVAILABLE:
    mean_shap = np.abs(shap_vals.values).mean(axis=0)
    importance_df = native_pd.DataFrame({
        "feature": feature_names,
        "importance": mean_shap,
    }).sort_values("importance", ascending=False)

    print(f"Top {SHAP_TOP_FEATURES} Most Important Features:")
    display_table(importance_df.head(SHAP_TOP_FEATURES))

[//]: # (cr:doc name='11_7_customer_browser' id=60dc7971)
## 11.7 Customer Browser

In [ ]:
# @cr:code name='build_customer_browser' id=603a1951
if _SCORING_AVAILABLE:
    browser_df = predictions_df.merge(
        scoring_features[[ENTITY_KEY] + feature_names],
        on=ENTITY_KEY,
        how="left",
    )

    print(f"Customer browser ready with {len(browser_df):,} records")
    print("\nPrediction Distribution:")
    print(f"  Predicted Positive: {(browser_df['prediction'] == 1).sum():,}")
    print(f"  Predicted Negative: {(browser_df['prediction'] == 0).sum():,}")
    print(f"\nCorrect Predictions: {browser_df['correct'].sum():,}/{len(browser_df):,} ({browser_df['correct'].mean():.1%})")

In [ ]:
# @cr:code name='define_show_customer' id=1b180992
if _SCORING_AVAILABLE:
    def _compute_entity_shap(x_row):
        sv = explainer(x_row)
        if len(sv.shape) == 3:
            return sv[0, :, 1]
        return sv[0]

    def show_customer(idx: int):
        row = browser_df.iloc[idx]
        entity_id = row[ENTITY_KEY]

        print(f"=== Customer {entity_id} ===")
        print(f"Prediction: {int(row['prediction'])} (probability: {row['probability']:.3f})")
        print(f"Actual: {int(row['actual'])}")
        print(f"Correct: {'Yes' if row['correct'] else 'No'}")
        print()

        feature_vals = X.iloc[idx]
        entity_shap = _compute_entity_shap(X.iloc[[idx]])

        feature_impact = native_pd.DataFrame({
            "feature": feature_names,
            "value": feature_vals.to_numpy(),
            "shap_impact": entity_shap.values,
        }).sort_values("shap_impact", key=abs, ascending=False)

        print("Top Contributing Features:")
        display_table(feature_impact.head(SHAP_TOP_FEATURES))

        plt.figure(figsize=(10, 6))
        shap.plots.waterfall(entity_shap, max_display=SHAP_TOP_FEATURES, show=False)
        plt.title(f"SHAP Explanation for Customer {entity_id}")
        plt.tight_layout()
        plt.show()

In [ ]:
# @cr:code name='display_sample_customers' id=23618483
if _SCORING_AVAILABLE:
    print("Showing first 3 customers:\n")
    for i in range(min(3, len(browser_df))):
        show_customer(i)
        print("\n" + "=" * 60 + "\n")

In [ ]:
# @cr:code name='define_lookup_customer' id=599c0a7b
if _SCORING_AVAILABLE:
    def lookup_customer(entity_id):
        mask = browser_df[ENTITY_KEY] == entity_id
        if not mask.any():
            print(f"Customer {entity_id} not found in scoring set")
            return
        idx = browser_df[mask].index[0]
        x_idx = browser_df.index.get_loc(idx)
        show_customer(x_idx)

    print("Available entity IDs (first 10):")
    print(browser_df[ENTITY_KEY].head(10).tolist())

[//]: # (cr:doc name='11_8_error_analysis' id=1d1e5406)
## 11.8 Error Analysis

In [ ]:
# @cr:code name='analyze_misclassified' id=4f161121
if _SCORING_AVAILABLE:
    incorrect = browser_df[browser_df["correct"] == 0]
    print(f"Misclassified customers: {len(incorrect):,}")

    fp = incorrect[incorrect["prediction"] == 1]
    print(f"  False Positives: {len(fp):,}")

    fn = incorrect[incorrect["prediction"] == 0]
    print(f"  False Negatives: {len(fn):,}")

In [ ]:
# @cr:code name='show_false_positive' id=111a00ad
if _SCORING_AVAILABLE and len(fp) > 0:
    print("\n=== Example False Positive ===")
    fp_idx = browser_df.index.get_loc(fp.index[0])
    show_customer(fp_idx)

In [ ]:
# @cr:code name='show_false_negative' id=59dc5c6d
if _SCORING_AVAILABLE and len(fn) > 0:
    print("\n=== Example False Negative ===")
    fn_idx = browser_df.index.get_loc(fn.index[0])
    show_customer(fn_idx)

[//]: # (cr:doc name='11_9_export_results' id=42718340)
## 11.9 Export Results

In [ ]:
# @cr:code name='save_scoring_results' id=3a43addb
if _SCORING_AVAILABLE:
    if is_databricks():
        from customer_retention.core.compat import normalize_timestamps, pandas_dtype_to_spark_schema
        from customer_retention.core.compat.detection import get_spark_session
        spark = get_spark_session()

    output_dir = config.scoring_output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    importance_df.to_csv(output_dir / "feature_importance.csv", index=False)
    print(f"Feature importance saved to {output_dir / 'feature_importance.csv'}")

    top_features = importance_df.head(SHAP_TOP_FEATURES)["feature"].tolist()
    shap_by_entity = native_pd.DataFrame({ENTITY_KEY: scoring_features[ENTITY_KEY].to_numpy()})

    _full_shap_vals = np.full((len(X), len(feature_names)), np.nan)
    _sample_vals = shap_vals.values if hasattr(shap_vals, "values") else np.asarray(shap_vals)
    _full_shap_vals[_shap_indices] = _sample_vals

    for feat in top_features:
        feat_idx = feature_names.index(feat)
        shap_by_entity[f"shap_{feat}"] = _full_shap_vals[:, feat_idx]

    detailed_df = predictions_df.merge(shap_by_entity, on=ENTITY_KEY, how="left")
    detailed_df.to_parquet(output_dir / "predictions_with_shap.parquet", index=False)
    print(f"Detailed predictions with SHAP saved to {output_dir / 'predictions_with_shap.parquet'}")
    print(f"  SHAP values present for {len(_shap_indices)}/{len(X)} entities (sampled subset)")

    if is_databricks():
        table_name = f"{config.catalog}.{config.schema}.scoring_results"
        normalized = normalize_timestamps(detailed_df)
        schema = pandas_dtype_to_spark_schema(normalized)
        spark.createDataFrame(normalized, schema=schema).write.format("delta") \
            .option("overwriteSchema", "true").mode("overwrite").saveAsTable(table_name)
        print(f"Results saved to Delta table: {table_name}")

In [ ]:
# @cr:code name='release_stage_memory' id=5e6d3db5
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
del features_df

[//]: # (cr:doc name='section' id=743f2aeb)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.